# Lead Scoring Model

This is a theoretical representation of the model; the parameters I will be using are:

1. Financial Qualification
2. Need, Problem and Product Fit
3. Authority and Decision Structure
4. Timeline, Urgency and Buying Stage
5. Engagement Behaviour
6. Company and Market Fit
7. Lead Source Quality
8. Competitive Landscape
9. Relationship and Trust Equity
10. Strategic and Lifetime Value

Within each parameter I introduce multiple variables to capture every potential scenario in the lead conversion journey. Each variable is assigned a specific weight to calculate a precise metric for that parameter. Finally, by weighting the parameters themselves, the model produces a robust, comprehensive score that accounts for most of the edge cases as well.

## Helper Functions

Before computing any parameter, we define a library of reusable mathematical tools that the scoring engine relies on throughout:

| Function | Purpose |
|---|---|
| `bci` (Bayesian Credible Interval) | Given observed successes and trials, returns a conservative lower-bound estimate from a Beta posterior, so that small sample sizes are automatically penalised. Used wherever a stated input is blended with historical evidence (budget confirmation, technical coverage, win rate, etc.). |
| `bci_from_rate` | Convenience wrapper that converts a rate plus sample size into a `bci` call. |
| `edp` (Exponential Decay Penalty) | Computes `exp(−k·x)` — a smooth, tuneable decay. Used for recency penalties (days since engagement), fiscal-year distance, compliance gaps, churn risk, and competitor-count suppression. |
| `ln_norm` (Logarithmic Normalisation) | Computes `log(1+count) / log(1+cap)` — maps an integer count into `[0, 1]` with diminishing returns. Used for stakeholder counts, engagement channel counts, use-case counts, and relationship tenure. |
| `sigmoid_transform` | A logistic curve centred at `x0` with steepness `k`. Compresses raw scores into a bounded `[0, 1]` range and amplifies separation around the decision boundary. Applied to budget ratio, engagement, company fit, and selected final scores. |
| `gaussian_penalty` | Computes `exp(−(x−ideal)² / 2σ²)` — penalises deviation from an ideal value with a bell-shaped curve, for variables where both "too high" and "too low" are undesirable. |
| `hrc` (Huber Robust Composite) | A robust weighted average: it computes an initial weighted mean, then iteratively down-weights any component whose value deviates from the composite by more than `δ`. This prevents a single outlier variable from disproportionately dragging a parameter score up or down. It is the primary aggregation function for every parameter. |
| `silverman_bandwidth` | Estimates the optimal kernel bandwidth from a historical score array using Silverman's rule. Used internally by `kpn`. |
| `kpn` (Kernel Percentile Normalisation) | Given a raw score and a distribution of historical scores, returns the percentile rank via Gaussian kernel density estimation, so a score can be interpreted relative to the population of past leads rather than in absolute terms. |
| `ewma_mean` (Exponentially Weighted Moving Average) | Weights recent observations more heavily than older ones (decay factor `λ`). Used to detect engagement momentum from a weekly activity series. |


In [168]:
import numpy as np
import math
from scipy.stats import beta as beta_dist, norm

def bci(successes, trials, confidence=0.05):
    if trials <= 0:
        return 0.0
    s = float(successes)
    n = float(trials)
    return beta_dist.ppf(confidence, s + 1, n - s + 1)
    
def bci_from_rate(rate, n):
    s = round(rate * n)
    return bci(s, n)
              
def edp(x, k):
    return np.exp(-k * x)
def ln_norm(count, cap):
    if cap <= 0:
        return 0.0
    return min(np.log(1 + count) / np.log(1 + cap), 1.0)
    
def sigmoid_transform(x, x0=0.50, k=8.0):
    return 1.0 / (1.0 + np.exp(-k * (x - x0)))
    
def gaussian_penalty(actual, ideal, sigma):
    return np.exp(-((actual - ideal) ** 2) / (2 * sigma ** 2))
    
def hrc(values, weights, delta=0.15, max_iter=100, tol=1e-6):
    values = np.array(values, dtype=float)
    weights = np.array(weights, dtype=float)
    weights = weights / weights.sum()
    theta = np.dot(weights, values)
    for _ in range(max_iter):
        residuals = np.abs(values - theta)
        adjusted_weights = weights.copy()
        for i in range(len(values)):
            if residuals[i] > delta:
                adjusted_weights[i] = weights[i] * (delta / residuals[i])
        if adjusted_weights.sum() == 0:
            break
        theta_new = np.dot(adjusted_weights, values) / adjusted_weights.sum()
        if abs(theta_new - theta) < tol:
            theta = theta_new
            break
        theta = theta_new
    return theta
    
def silverman_bandwidth(history):
    history = np.asarray(history, dtype=float)
    n = len(history)
    if n < 2:
        return 1.0
    sigma = np.std(history, ddof=1)
    q75, q25 = np.percentile(history, [75, 25])
    iqr = q75 - q25
    spread = min(sigma, iqr / 1.34) if iqr > 0 else sigma
    if spread <= 0:
        return 1.0
    return 0.9 * spread * (n ** (-1 / 5))
    
def kpn(raw_score, historical_scores):
    historical_scores = np.asarray(historical_scores, dtype=float)
    n = len(historical_scores)
    if n < 2:
        return raw_score 
    h = silverman_bandwidth(historical_scores)
    if h <= 0:
        return 0.5
    z = (raw_score - historical_scores) / h
    return float(np.mean(norm.cdf(z)))
    
def ewma_mean(values, lam=0.85):
    values = np.asarray(values, dtype=float)
    T = len(values)
    weights = np.array([lam ** (T - t - 1) for t in range(T)])
    weights /= weights.sum()
    return float(np.sum(weights * values))

In [153]:
data = [
    # ═══════════════════════════════════════════════════════
    # PARAMETER 1 — FINANCIAL QUALIFICATION
    # ═══════════════════════════════════════════════════════
    {
        "b": 50000,         # Estimated budget
        "d": 45000,         # Deal value (our price)
        "c": 0.75,          # Budget confirmation level [0,1]
        "t": 3,             # Months until fiscal year alignment
        "f2": 0.7,          # Funding source type [0,1]
        "m": 0.5,           # Multi-year willingness [0,1]
        "p": 0.75,          # Procurement complexity [0.5,1]
        "n_similar_deals": 12,   # how many similar-sized deals we've quoted
        "n_confirmed_budget": 8, # how many actually had confirmed budget
    },
    # ═══════════════════════════════════════════════════════
    # PARAMETER 2 — NEED, PROBLEM & PRODUCT FIT
    # ═══════════════════════════════════════════════════════
    {
        "r": 0.8,           # Industry match [0,1]
        "s": 0.9,           # Solution fit [0,1]
        "p": 0.7,           # Problem clarity [0,1]
        "cs": 1,            # Current solution exists {0, 0.5, 1}
        "d": 0.6,           # Dissatisfaction with current [0,1]
        "t": 0.85,          # Technical requirements coverage [0,1]
        "c": 0.2,           # Custom work needed [0,1]
        "c2": 0.8,          # Compliance fit {0, 0.5, 1}
        "u": 3,             # Number of use cases articulated
        "n_features_needed": 20,
        "n_features_available": 17,
    },
    # ═══════════════════════════════════════════════════════
    # PARAMETER 3 — AUTHORITY & DECISION STRUCTURE
    # ═══════════════════════════════════════════════════════
    {
        "r": 0.8,           # Contact role seniority
        "d": 1,             # Decision involvement {0, 0.5, 1}
        "n": 4,             # Number of stakeholders identified
        "o": 0.8,           # Org alignment [0,1]
        "p": 0.7,           # Direct access to senior stakeholders [0,1]
        "n_stakeholders_met": 4,
        "n_stakeholders_supportive": 3,
    },
    # ═══════════════════════════════════════════════════════
    # PARAMETER 4 — TIMELINE, URGENCY & BUYING STAGE
    # ═══════════════════════════════════════════════════════
    {
        "t": 45,            # Days until decision
        "t1": 1,            # Trigger event happened {0,1}
        "t2": 30,           # Days until trigger deadline
        "ep": 0.8,          # Evaluation process maturity [0,1]
        "ns": 1,            # Next step defined {0,1}
        "cp": 0.3,          # Competing priorities [0,1]
    },
    # ═══════════════════════════════════════════════════════
    # PARAMETER 5 — ENGAGEMENT BEHAVIOUR
    # ═══════════════════════════════════════════════════════
    {
        "n1": 10, "n2": 5, "n3": 3, "n4": 4, "n5": 8,
        "n6": 2, "n7": 3, "n8": 1, "n9": 2,
        "t": 7,             # Days since last engagement
        "v": 1.5,           # Velocity ratio (last 14d / prior 14d)
        "c": 4,             # Channel diversity count
        "m": 1,             # Negative signal count
        "weekly_engagement": [0.3, 0.4, 0.5, 0.55, 0.7, 0.8, 0.85, 0.9],
    },
    # ═══════════════════════════════════════════════════════
    # PARAMETER 6 — COMPANY & MARKET FIT
    # ═══════════════════════════════════════════════════════
    {
        "seg": 0.9, "emp": 0.8, "rev": 0.7, "tech": 0.85,
        "geo": 1.0, "gro": 0.7, "f": 0.8, "d": 0.75, "l": 0.9,
    },
    # ═══════════════════════════════════════════════════════
    # PARAMETER 7 — LEAD SOURCE QUALITY
    # ═══════════════════════════════════════════════════════
    {
        "q": 0.75, "p": 0.8, "r": 0.7, "s": 1.0,
        "n_leads_from_source": 40,
        "n_converted_from_source": 22,
    },
    # ═══════════════════════════════════════════════════════
    # PARAMETER 8 — COMPETITIVE LANDSCAPE
    # ═══════════════════════════════════════════════════════
    {
        "n": 3,             # Number of competitors
        "w": 0.6,           # Historical win rate [0,1]
        "s": 0.3,           # Incumbent strength [0,1]
        "d": 0.8,           # Differentiation [0,1]
        "c": 0.4,           # Switching cost [0,1]
        "k": 0,             # Sole vendor? {0,1}
        "n_competitive_deals": 30,
        "n_wins": 18,
    },
    # ═══════════════════════════════════════════════════════
    # PARAMETER 9 — RELATIONSHIP & TRUST EQUITY
    # ═══════════════════════════════════════════════════════
    {
        "pb": 1.0, "rt": 18, "nps": 0.5, "es": 0.5,
        "t": 0.7, "rc": 0.8, "n": 0,
    },
    # ═══════════════════════════════════════════════════════
    # PARAMETER 10 — STRATEGIC & LIFETIME VALUE
    # ═══════════════════════════════════════════════════════
    {
        "ltv": 1500000, "cac": 400000, "cs": 0.7, "b": 0.6,
        "m": 0.5, "e": 0.65, "c": 0.3,
        "n_similar_accounts": 15,
        "n_expanded": 9,
    },
]

gates = ["f", "f", "f", "f", "f", "f", "f"]
modifiers = [1, 1, 0.9, 1, 1]


## Parameter 1 — Financial Qualification

This parameter assesses whether the prospect can realistically pay for the deal. It combines budget capacity, confirmation reliability, fiscal timing, funding security, long-term willingness, and procurement friction.

### Variables

| Symbol | Description | Range |
|---|---|---|
| `B` | Estimated budget — the total amount the prospect has allocated or can realistically allocate for our solution. | currency |
| `D` | Deal value — our total quoted price (licence, implementation, subscription, fees). | currency |
| `Br` | Budget ratio = `min(B/D, 1)`. Ideal value → 1. | `[0, 1]` |
| `C` | Budget confirmation level — reliability of the budget figure. 0 = unconfirmed, 0.25 = range mentioned, 0.5 = verbal, 0.75 = written/email, 1 = formal procurement document. | `[0, 1]` |
| `F1` | Fiscal year alignment — how far out the prospect's buying cycle sits, computed via exponential decay `exp(−0.25·T_months)` so alignment degrades smoothly with distance. | `[0, 1]` |
| `F2` | Funding source type — security of the funding source. 0.3 = unidentified, 0.5 = departmental discretionary, 0.7 = allocated project budget, 1.0 = board-approved CapEx. | `[0, 1]` |
| `M` | Multi-year willingness — openness to contracts longer than one year. 0 = refuses, 0.5 = conditional, 1 = actively seeking multi-year. | `[0, 1]` |
| `P` | Procurement complexity — difficulty of the approval process. 1.0 = credit-card simple, 0.75 = MSA + PO, 0.5 = full RFP with committee sign-offs. | `[0.5, 1]` |

### How it is computed

- **Budget ratio** is passed through a sigmoid transform (`x0 = 0.70`, `k = 10`). This compresses ratios near 1.0 (diminishing returns once budget clearly covers the deal) and penalises ratios below ~0.70 more aggressively.
- **Budget confirmation** blends the stated level `C_stated` 50/50 with a Bayesian credible interval `C_evidence`, derived from `n_confirmed_budget` out of `n_similar_deals`. This anchors confidence to empirical conversion history rather than self-reported confirmation alone.
- **Fiscal alignment** uses exponential decay `exp(−0.25·T)`, giving a smooth degradation that never goes negative.
- All six components are aggregated with the Huber Robust Composite (`hrc`), so a single outlier variable cannot disproportionately inflate or deflate the parameter score.

### Composite Formula

```
X1 = HRC(
    values  = [Br_sigmoid, C_blended, F1_decay, F2, M, P],
    weights = [0.45, 0.17, 0.12, 0.12, 0.08, 0.06]
)
```


In [196]:
p1 = data[0]

br_raw = min(p1["b"] / p1["d"], 1.0)
Br = sigmoid_transform(br_raw, x0=0.70, k=10.0)

C_stated = p1["c"]
C_evidence = bci(p1["n_confirmed_budget"], p1["n_similar_deals"])
C = 0.5 * C_stated + 0.5 * C_evidence

F1 = edp(p1["t"], 0.25)

F2 = p1["f2"]

M_val = p1["m"]

P_val = p1["p"]

X1 = hrc(
    values=[Br, C, F1, F2, M_val, P_val],
    weights=[0.45, 0.17, 0.12, 0.12, 0.08, 0.06]
)

print(f"X1 (Financial Qualification): {X1:.6f}")


X1 (Financial Qualification): 0.783330


## Parameter 2 — Need, Problem & Product Fit

Evaluates how well our product matches the prospect's needs, technical environment, and compliance requirements.

### Variables

| Symbol | Description | Range |
|---|---|---|
| `R` | Industry match — alignment with industries where we have proven success. | `[0, 1]` |
| `S` | Solution fit — how directly our product solves their stated problem. | `[0, 1]` |
| `P` | Problem clarity — how well the prospect can articulate the problem. 0 = vague, 1 = detailed and documented. | `[0, 1]` |
| `Cs` | Current solution exists — whether they already use a competing method. 0 = none, 0.5 = partial, 1 = full alternative in place. | `{0, 0.5, 1}` |
| `D` | Dissatisfaction with current solution. 0 = fully satisfied, 1 = deeply dissatisfied. | `[0, 1]` |
| `T` | Technical requirements coverage — share of stated requirements our product handles out-of-the-box. | `[0, 1]` |
| `C` | Custom work needed — extent of additional development or configuration required. | `[0, 1]` |
| `C2` | Compliance fit — whether our product meets the prospect's regulatory requirements. 0 = non-compliant, 0.5 = partial, 1 = fully compliant. | `{0, 0.5, 1}` |
| `U` | Number of distinct use cases the prospect has articulated (proxy for buyer intent and deal maturity). | integer |

### How it is computed

- **Technical coverage** (`T`) is computed via BCI from `n_features_available` out of `n_features_needed`, penalising high-coverage claims when the feature sample is small.
- **Positive fit** sub-score `F_pos` aggregates `[R, S, P, T_bci]` using `hrc` (weights 0.25, 0.30, 0.20, 0.25).
- **Negative fit** sub-score `F_neg` combines three drag terms via `hrc` (weights 0.40, 0.25, 0.35):
  - Dissatisfaction drag: `1 − 0.4·D·Cs`
  - Custom drag: `1 − 0.25·C`
  - Compliance gap via exponential decay `exp(−3.0·(1 − C2))` — a small gap barely hurts, a full gap is devastating.
- **Use-case count** is normalised via `ln_norm(U, 6)` (diminishing returns past 6).
- The three sub-scores are combined with `hrc` (weights 0.45, 0.40, 0.15).

### Composite Formula

```
F_pos   = HRC([R, S, P, T_bci], [0.25, 0.30, 0.20, 0.25])
F_neg   = HRC([1 − 0.4·D·Cs, 1 − 0.25·C, exp(−3·(1−C2))], [0.40, 0.25, 0.35])
U_score = ln_norm(U, 6)
X2      = HRC([F_pos, F_neg, U_score], [0.45, 0.40, 0.15])
```


In [185]:
p2 = data[1]

R_ind = p2["r"]
S_fit = p2["s"]
P_clar = p2["p"]

T_tech = bci(p2["n_features_available"], p2["n_features_needed"])

F_pos = hrc(
    values=[R_ind, S_fit, P_clar, T_tech],
    weights=[0.25, 0.30, 0.20, 0.25]
)

dissatisfaction_drag = p2["d"] * p2["cs"] 
custom_drag = p2["c"]

compliance_gap = 1.0 - p2["c2"]
compliance_penalty = edp(compliance_gap, 3.0)

F_neg = hrc(
    values=[1.0 - 0.4 * dissatisfaction_drag,
            1.0 - 0.25 * custom_drag,
            compliance_penalty],
    weights=[0.40, 0.25, 0.35]
)

U_score = ln_norm(p2["u"], 6)

X2 = hrc(
    values=[F_pos, F_neg, U_score],
    weights=[0.45, 0.40, 0.15]
)

print(f"X2 (Need & Product Fit): {X2:.6f}")


X2 (Need & Product Fit): 0.745826


## Parameter 3 — Authority & Decision Structure

Measures who is involved in the buying decision, how much authority they carry, and how aligned the internal stakeholders are.

### Variables

| Symbol | Description | Range |
|---|---|---|
| `R` | Primary contact seniority. | `{0.2, 0.4, 0.6, 0.8, 1}` |
| `D` | Decision involvement of the contact. 0 = none, 0.5 = influencer, 1 = final decision-maker. | `{0, 0.5, 1}` |
| `Ac` | Contact authority = `R × D`. | `[0, 1]` |
| `N` | Total stakeholders identified in the purchasing decision. | integer |
| `O` | Organisational alignment — whether multiple departments agree on the need. | `[0, 1]` |
| `P` | Direct access to senior stakeholders. | `[0, 1]` |

### How it is computed

- **Stakeholder count** is normalised via `ln_norm(N, 5)` (log-based, diminishing returns past 5).
- **Organisational alignment** blends the stated `O` 50/50 with a BCI score from `n_stakeholders_supportive` out of `n_stakeholders_met`, anchoring stated alignment to observed stakeholder buy-in.
- All four components are aggregated via `hrc` (weights 0.30, 0.30, 0.25, 0.15).

### Composite Formula

```
Ac         = R × D
N_score    = ln_norm(N, 5)
O_combined = 0.5·O_stated + 0.5·BCI(supportive, met)
X3         = HRC([Ac, N_score, O_combined, P], [0.30, 0.30, 0.25, 0.15])
```


In [186]:
p3 = data[2]

A_c = p3["r"] * p3["d"]

N_stake = ln_norm(p3["n"], 5)

O_stated = p3["o"]
O_evidence = bci(p3["n_stakeholders_supportive"], p3["n_stakeholders_met"])
O_combined = 0.5 * O_stated + 0.5 * O_evidence

P_access = p3["p"]

X3 = hrc(
    values=[A_c, N_stake, O_combined, P_access],
    weights=[0.30, 0.30, 0.25, 0.15]
)

print(f"X3 (Authority & Decision): {X3:.6f}")


X3 (Authority & Decision): 0.769298


## Parameter 4 — Timeline, Urgency & Buying Stage

Measures how soon the prospect will decide and how much momentum the deal carries.

### Variables

| Symbol | Description | Range |
|---|---|---|
| `T` | Days until expected final decision. | integer |
| `T1` | Whether a trigger event has occurred. | `{0, 1}` |
| `T2` | Days until the trigger event deadline. | integer |
| `Ep` | Evaluation process maturity — how structured the buying process is. | `[0, 1]` |
| `Ns` | Next step defined — whether a concrete next action is scheduled. | `{0, 1}` |
| `Cp` | Competing priorities — how much attention is divided elsewhere. 0 = top priority, 1 = heavily distracted. | `[0, 1]` |

### How it is computed

- **Timeline decay** (`T'`) uses exponential decay `exp(−0.015·T)`: deals at 0 days score ~1.0, at 45 days ~0.51, and distant deals asymptote toward 0 on a single smooth curve.
- **Trigger urgency** (`U_trig`) uses `T1 · exp(−0.01·T2)`, giving a smooth, always-positive urgency signal when a trigger exists (0 otherwise).
- **Process maturity** combines evaluation maturity and a defined next step: `0.5·Ep + 0.5·Ns`.
- **Competing priorities** enter the aggregation as the component `1 − Cp`, so distraction risk interacts with the other timing signals through the robust composite rather than acting as a blanket scalar.
- All four components are aggregated via `hrc` (weights 0.45, 0.15, 0.25, 0.15).

### Composite Formula

```
T'        = exp(−0.015·T)
U_trig    = T1 · exp(−0.01·T2)     [0 if no trigger]
P_process = 0.5·Ep + 0.5·Ns
X4        = HRC([T', U_trig, P_process, 1 − Cp], [0.45, 0.15, 0.25, 0.15])
```


In [187]:
p4 = data[3]

T_prime = edp(p4["t"], 0.015)

if p4["t1"] == 1:
    U_trig = edp(p4["t2"], 0.01) 
else:
    U_trig = 0.0

P_proc = 0.5 * p4["ep"] + 0.5 * p4["ns"]

T_core = hrc(
    values=[T_prime, U_trig, P_proc, 1.0 - p4["cp"]], 
    weights=[0.45, 0.15, 0.25, 0.15]
)

X4 = T_core

print(f"X4 (Timeline & Urgency): {X4:.6f}")


X4 (Timeline & Urgency): 0.643658


## Parameter 5 — Engagement Behaviour

Measures how actively the prospect is interacting across channels and whether that activity is accelerating or decaying.

### Variables

| Symbol | Description | Range |
|---|---|---|
| `N1`–`N9` | Activity counts across nine channels: email opens (`N1`), email replies (`N2`), meetings held (`N3`), calls (`N4`), web sessions (`N5`), downloads (`N6`), pricing page visits (`N7`), demo requests (`N8`), social/community (`N9`). | integer each |
| `T` | Days since last meaningful interaction. | integer |
| `V` | Engagement velocity — ratio of last-14-day engagements to prior 14 days. | float |
| `C` | Channel diversity — number of distinct channels used. | integer |
| `M` | Negative signal count (unsubscribes, cancellations, no-shows, etc.). | integer |
| `weekly_engagement` | Time series of weekly engagement scores, oldest to newest, used for momentum detection. | list of floats |

### How it is computed

- **Per-channel normalisation**: each of the nine activity counts is independently normalised via `ln_norm(Ni, cap_i)` with channel-specific caps (e.g. 20 for email opens, 3 for demo requests). This prevents any single high-volume channel from dominating and ensures diminishing returns.
- **Composite engagement**: the nine normalised channel scores are aggregated via `hrc` with weights reflecting signal strength:
  - Demo requests and meetings: 0.18 each (strongest buying signals)
  - Pricing visits: 0.14
  - Email replies, downloads, calls: 0.10–0.12
  - Web sessions, email opens, social: 0.05–0.08
- **Recency decay**: `R_decay = exp(−0.033·T)`, applied multiplicatively to the composite engagement.
- **Momentum**: an EWMA (`λ = 0.85`) over `weekly_engagement` is compared to its simple mean; the positive difference (capped at 1.0) becomes the velocity bonus, scaled by 0.15, capturing genuine acceleration patterns.
- **Channel diversity**: `ln_norm(C, 5) × 0.10`.
- **Negative signals**: penalised via `exp(−0.3·M)`, applied as `√(N_penalty)` to soften impact while still discouraging repeated negative signals.
- **Final assembly**: core engagement (composite × decay), velocity bonus, and channel diversity are combined via `hrc` (weights 0.75, 0.15, 0.10), multiplied by `√(N_penalty)`, then passed through a sigmoid (`x0 = 0.3`, `k = 8`) for the final bounded score.

### Composite Formula

```
e_i          = ln_norm(N_i, cap_i)            for each channel i
E_composite  = HRC([e1 … e9], [0.05, 0.10, 0.18, 0.12, 0.08, 0.10, 0.14, 0.18, 0.05])
R_decay      = exp(−0.033·T)
V_bonus      = min(max(0, (EWMA − mean) / mean), 1.0) × 0.15
D_channel    = ln_norm(C, 5) × 0.10
N_penalty    = exp(−0.3·M)
E_core       = HRC([E_composite·R_decay, V_bonus, D_channel], [0.75, 0.15, 0.10])
X5           = sigmoid(E_core·√N_penalty, x0=0.3, k=8)
```


In [188]:
p5 = data[4]

e_email_open  = ln_norm(p5["n1"], 20) 
e_email_reply = ln_norm(p5["n2"], 10)  
e_meetings    = ln_norm(p5["n3"], 6) 
e_calls       = ln_norm(p5["n4"], 8)
e_web         = ln_norm(p5["n5"], 15) 
e_downloads   = ln_norm(p5["n6"], 8) 
e_pricing     = ln_norm(p5["n7"], 5) 
e_demo_req    = ln_norm(p5["n8"], 3) 
e_social      = ln_norm(p5["n9"], 5) 


E_composite = hrc(
    values=[e_email_open, e_email_reply, e_meetings, e_calls,
            e_web, e_downloads, e_pricing, e_demo_req, e_social],
    weights=[0.05, 0.10, 0.18, 0.12, 0.08, 0.10, 0.14, 0.18, 0.05]
)

R_decay = edp(p5["t"], 0.033)

weekly = p5["weekly_engagement"]
ewma_val = ewma_mean(weekly, lam=0.85)
simple_mean = np.mean(weekly)

V_bonus = max(0.0, (ewma_val - simple_mean) / max(simple_mean, 0.01))
V_score = min(V_bonus, 1.0) * 0.15

D_channel = ln_norm(p5["c"], 5) * 0.10

N_penalty = edp(p5["m"], 0.3)

E_core = hrc(
    values=[E_composite * R_decay, V_score, D_channel],
    weights=[0.75, 0.15, 0.10]
)

X5_raw = E_core * np.sqrt(N_penalty)
X5 = sigmoid_transform(X5_raw, x0=0.3, k=8.0)

print(f"X5 (Engagement Behaviour): {X5:.6f}")


X5 (Engagement Behaviour): 0.727083


## Parameter 6 — Company & Market Fit

Measures Ideal Customer Profile (ICP) alignment — how closely the prospect matches the type of customer our company serves best, and how practical the business relationship would be.

### Variables

| Symbol | Description | Range |
|---|---|---|
| `Seg` | Customer segment match. | `[0, 1]` |
| `Emp` | Employee size fit relative to ideal. | `[0, 1]` |
| `Rev` | Annual revenue fit relative to ideal range. | `[0, 1]` |
| `Tech` | Technology stack compatibility. | `[0, 1]` |
| `Geo` | Geographic alignment with active service regions. | `[0, 1]` |
| `Gro` | Company growth trajectory. | `[0, 1]` |
| `F` | Financial health / credit risk. | `[0, 1]` |
| `D` | Digital adoption readiness. | `[0, 1]` |
| `L` | Language and culture compatibility. | `[0, 1]` |

### How it is computed

- The nine ICP variables are aggregated via `hrc` (weights 0.20, 0.10, 0.10, 0.20, 0.05, 0.10, 0.10, 0.10, 0.05), providing robustness against any single outlier dimension.
- The result is passed through a sigmoid (`x0 = 0.70`, `k = 8`) to sharpen discrimination around the "good-fit" boundary: prospects solidly above 0.70 are rewarded, those below are penalised more steeply than a linear score would reflect.

### Composite Formula

```
X6_raw = HRC([Seg, Emp, Rev, Tech, Geo, Gro, F, D, L],
             [0.20, 0.10, 0.10, 0.20, 0.05, 0.10, 0.10, 0.10, 0.05])
X6     = sigmoid(X6_raw, x0=0.70, k=8)
```


In [189]:

p6 = data[5]

X6_raw = hrc(
    values=[p6["seg"], p6["emp"], p6["rev"], p6["tech"],
            p6["geo"], p6["gro"], p6["f"], p6["d"], p6["l"]],
    weights=[0.20, 0.10, 0.10, 0.20, 0.05, 0.10, 0.10, 0.10, 0.05]
)

X6 = sigmoid_transform(X6_raw, x0=0.70, k=8.0)

print(f"X6 (Company & Market Fit): {X6:.6f}")


X6 (Company & Market Fit): 0.720586


## Parameter 7 — Lead Source Quality

Evaluates where the lead came from, how that source channel historically performs, and how much useful data we captured at entry.

### Variables

| Symbol | Description | Range |
|---|---|---|
| `Q` | Source channel quality — historical effectiveness of the entry channel (e.g. 0.15 for purchased lists up to 0.90 for customer referrals). | `[0, 1]` |
| `P` | Campaign/asset quality — performance of the specific campaign relative to best-performing. | `[0, 1]` |
| `R` | Data richness at entry. 0 = email only, 1 = full profile with company, role, phone, stated interest. | `[0, 1]` |
| `S` | Inbound vs. outbound. 0.4 = outbound (we initiated), 1.0 = inbound (they came to us). | `{0.4, 1}` |

### How it is computed

- **Source channel quality** (`Q`) blends the stated quality 50/50 with a BCI score from `n_converted_from_source` out of `n_leads_from_source`. A channel with a strong conversion history on a decent sample validates the stated quality; a thin sample keeps the BCI conservative and pulls the score down.
- The four components are aggregated via `hrc` (weights 0.55, 0.15, 0.10, 0.20).

### Composite Formula

```
Q  = 0.5·Q_stated + 0.5·BCI(converted, total_leads)
X7 = HRC([Q, P, R, S], [0.55, 0.15, 0.10, 0.20])
```


In [190]:
p7 = data[6]

Q_stated = p7["q"]
Q_evidence = bci(p7["n_converted_from_source"], p7["n_leads_from_source"])
Q = 0.5 * Q_stated + 0.5 * Q_evidence

P_camp = p7["p"]
R_data = p7["r"]
S_dir  = p7["s"]

X7 = hrc(
    values=[Q, P_camp, R_data, S_dir],
    weights=[0.55, 0.15, 0.10, 0.20]
)

print(f"X7 (Lead Source Quality): {X7:.6f}")


X7 (Lead Source Quality): 0.677632


## Parameter 8 — Competitive Landscape

Evaluates competitive pressure — how many alternatives the prospect is considering, our historical win rate, incumbent strength, differentiation, and switching friction.

### Variables

| Symbol | Description | Range |
|---|---|---|
| `N` | Number of competing vendors being evaluated. | integer |
| `W` | Historical win rate against these competitors in this segment. | `[0, 1]` |
| `S` | Incumbent strength — how entrenched the current vendor is. 0 = weak/none, 1 = deeply entrenched. | `[0, 1]` |
| `D` | Differentiation — how clearly we stand out from alternatives. | `[0, 1]` |
| `C` | Switching cost for the prospect. 0 = trivial, 1 = massive. | `[0, 1]` |
| `K` | Sole vendor flag. 1 = we are the only vendor being considered. | `{0, 1}` |

### How it is computed

- If `K = 1` (sole vendor), the score is set directly to **1.0**.
- Otherwise:
  - **Competitor count** is suppressed via exponential decay `exp(−0.18·N)`, penalising the first few additional competitors steeply and flattening for crowded evaluations.
  - **Win rate** is computed via BCI from `n_wins` out of `n_competitive_deals`, grounding it in actual outcomes and staying conservative on small samples.
  - **Competitive drag** (incumbent strength + switching cost) uses `exp(−1.0·(0.5·S + 0.5·C))`: moderate friction has limited impact, extreme friction is nearly disqualifying.
  - All four components are aggregated via `hrc` (weights 0.25, 0.30, 0.20, 0.25).

### Composite Formula

```
If K = 1:  X8 = 1.0
Else:
    N_comp = exp(−0.18·N)
    W_bci  = BCI(wins, competitive_deals)
    Cd     = exp(−1.0·(0.5·S + 0.5·C))
    X8     = HRC([N_comp, W_bci, Cd, D], [0.25, 0.30, 0.20, 0.25])
```


In [191]:

p8 = data[7]

if p8["k"] == 1:
    X8 = 1.0
else:
    N_comp = edp(p8["n"], 0.18)

    W_bci = bci(p8["n_wins"], p8["n_competitive_deals"])

    friction = 0.5 * p8["s"] + 0.5 * p8["c"]
    Cd = edp(friction, 1.0)

    D_diff = p8["d"]

    X8 = hrc(
        values=[N_comp, W_bci, Cd, D_diff],
        weights=[0.25, 0.30, 0.20, 0.25]
    )

print(f"X8 (Competitive Landscape): {X8:.6f}")

X8 (Competitive Landscape): 0.620278


## Parameter 9 — Relationship & Trust Equity

Measures the strength and depth of the existing relationship between our organisation and the prospect — familiarity, trust signals, executive sponsorship, and any negative history.

### Variables

| Symbol | Description | Range |
|---|---|---|
| `Pb` | Previous business relationship. 0 = net new, 0.5 = lapsed/informal, 1.0 = active/recent customer. | `{0, 0.5, 1}` |
| `Rt` | Relationship tenure in months. | integer |
| `Nps` | Prospect sentiment. −1 = detractor, 0 = passive, 1 = promoter. New leads default to 0. | `[−1, 1]` |
| `Es` | Executive sponsor relationship. 0 = none, 0.5 = acquaintance, 1 = strong personal relationship. | `{0, 0.5, 1}` |
| `T` | Trust indicators — observable signals of transparency and collaboration. | `[0, 1]` |
| `Rc` | Reference customer — whether we can point to a relevant success story known to this prospect. | `[0, 1]` |
| `N` | Previous negative experience severity. 0 = none, 0.5 = minor, 1 = serious. | `{0, 0.5, 1}` |

### How it is computed

- **Relationship tenure** uses `Pb · ln_norm(Rt, 36)` — log-based normalisation with diminishing returns past 36 months, gated by whether a prior relationship exists at all.
- **Trust** sub-score aggregates `[Es, T, Rc, max(Nps, 0)]` via `hrc` (weights 0.30, 0.30, 0.20, 0.20).
- **Negative experience** penalty uses exponential decay `exp(−1.5·N)`: a minor issue (`N = 0.5`) still allows most of the score through, while a serious issue (`N = 1`) is nearly devastating.
- Tenure and trust are combined via `hrc` (weights 0.40, 0.60) before the negative penalty is applied multiplicatively.

### Composite Formula

```
R_ten   = Pb · ln_norm(Rt, 36)
R_trust = HRC([Es, T, Rc, max(Nps, 0)], [0.30, 0.30, 0.20, 0.20])
R_neg   = exp(−1.5·N)
X9      = R_neg · HRC([R_ten, R_trust], [0.40, 0.60])
```


In [192]:
p9 = data[8]

R_ten = p9["pb"] * ln_norm(p9["rt"], 36)

R_trust = hrc(
    values=[p9["es"], p9["t"], p9["rc"], max(p9["nps"], 0)],
    weights=[0.30, 0.30, 0.20, 0.20]
)

R_neg = edp(p9["n"], 1.5)

X9 = R_neg * hrc(
    values=[R_ten, R_trust],
    weights=[0.40, 0.60]
)

print(f"X9 (Relationship & Trust): {X9:.6f}")


X9 (Relationship & Trust): 0.693671


## Parameter 10 — Strategic & Lifetime Value

Evaluates the long-term value of the account beyond the immediate deal — lifetime revenue potential, strategic market impact, expansion likelihood, and churn risk.

### Variables

| Symbol | Description | Range |
|---|---|---|
| `LTV` | Estimated lifetime value of the customer relationship. | currency |
| `CAC` | Customer acquisition cost. | currency |
| `CAC'` | LTV / CAC ratio. | float |
| `Cs` | Cross-sell / up-sell opportunity after initial purchase. | `[0, 1]` |
| `B` | Brand value — reputational benefit of winning this account. | `[0, 1]` |
| `M` | Market expansion — whether this win opens a new market. 0 = no, 0.5 = strengthens existing, 1 = new market entry. | `{0, 0.5, 1}` |
| `E` | Expansion revenue probability — likelihood the account grows post-purchase. | `[0, 1]` |
| `C` | Churn risk — predicted probability of cancellation or non-renewal. | `[0, 1]` |

### How it is computed

- **LTV/CAC ratio** is passed through a sigmoid `σ(CAC', x0=3.0, k=1.5)`, rewarding healthy return-on-acquisition.
- **Cross-sell** score blends the stated value 50/50 with a BCI from `n_expanded` out of `n_similar_accounts`, grounding the estimate in historical expansion data for comparable accounts.
- **Strategic value** aggregates `[B, M, E]` via `hrc` (weights 0.30, 0.40, 0.30).
- **Churn penalty** uses exponential decay `exp(−1.2·C)`: moderate risk (`C = 0.3`) barely hurts, high risk (`C = 0.8`) slashes the score.
- The three value components are combined via `hrc` (weights 0.35, 0.25, 0.40) before the churn penalty is applied multiplicatively.

### Composite Formula

```
V_ltv   = sigmoid(LTV/CAC, x0=3.0, k=1.5)
Cs      = 0.5·Cs_stated + 0.5·BCI(expanded, similar_accounts)
V_strat = HRC([B, M, E], [0.30, 0.40, 0.30])
V_churn = exp(−1.2·C)
X10     = V_churn · HRC([V_ltv, Cs, V_strat], [0.35, 0.25, 0.40])
```


In [193]:
p10 = data[9]

cac_ratio = p10["ltv"] / p10["cac"]
V_ltv = sigmoid_transform(cac_ratio, x0=3.0, k=1.5)

Cs_stated = p10["cs"]
Cs_evidence = bci(p10["n_expanded"], p10["n_similar_accounts"])
Cs = 0.5 * Cs_stated + 0.5 * Cs_evidence

V_strat = hrc(
    values=[p10["b"], p10["m"], p10["e"]],
    weights=[0.30, 0.40, 0.30]
)

V_churn = edp(p10["c"], 1.2)

X10 = V_churn * hrc(
    values=[V_ltv, Cs, V_strat],
    weights=[0.35, 0.25, 0.40]
)

print(f"X10 (Strategic & Lifetime Value): {X10:.6f}")

X10 (Strategic & Lifetime Value): 0.439952


## Hard Knockout Gates

Seven binary gates that can instantly zero the entire lead score. If any single gate fails, the lead is disqualified regardless of how strong the other parameters are.

| Gate | Condition that fails the gate (sets it to 0) |
|---|---|
| `G1` | No realistic financial ability — confirmed budget = 0 **and** no identified funding path or budget cycle. |
| `G2` | Communication compliance violation — contact has opted out, unsubscribed, or sent a cease-and-desist. |
| `G3` | Pipeline integrity — lead is a duplicate record or already an active customer (should route to account management). |
| `G4` | Sanctions / compliance blacklist — company or individual appears on a sanctions list, trade restriction, or compliance blacklist. |
| `G5` | Geographic restriction — company is in a country or region we legally cannot sell to or support. |
| `G6` | Prohibited industry — company operates in an industry our organisation has a formal policy against serving. |
| `G7` | Technical infeasibility — after assessment, our product fundamentally cannot serve the prospect's core need and no roadmap path exists. |

Each gate is encoded as `"f"` (pass) or `"t"` (triggered / fail) in the `gates` list. The composite gate is:

```
G = G1 × G2 × G3 × G4 × G5 × G6 × G7
```

Any single 0 makes `G = 0`, killing the final score.


In [194]:
g1=[]
for i in range(len(gates)):
    h=gates[i]
    if h == 't':
        g1.append(0)
    else:
        g1.append(1)
G=1
for h in g1:
    G=G*h
print(G)

1


## Deal-Level Modifiers

Five multiplicative factors that adjust the final score for practical deal-execution realities. Unlike gates (binary), modifiers are continuous scalars that reduce the score proportionally.

| Modifier | What it captures | Range |
|---|---|---|
| `M1` | Data reliability — are key deal details (budget, company size, contacts, requirements) confirmed or estimated/missing? `0.5 + 0.5·data_completeness`. | `[0.5, 1.0]` |
| `M2` | Sales capacity — does the team have bandwidth and the right rep available? 1.0 = yes, 0.7 = stretched, 0.5 = no rep in region/segment. | `[0.5, 1.0]` |
| `M3` | Legal / contract complexity — severity of non-standard terms (liability clauses, IP terms, special agreements). 1.0 = standard, down to 0.6 for extreme requirements. | `[0.6, 1.0]` |
| `M4` | Payment / financial risk — currency fluctuation, long payment cycles, high-risk markets. 1.0 = low risk, down to 0.7. | `[0.7, 1.0]` |
| `M5` | Execution complexity — multi-location, language barriers, heavy customisation, complex approval chains. 1.0 = simple, down to 0.6 for very complex deals. | `[0.6, 1.0]` |

```
M = M1 × M2 × M3 × M4 × M5
```


In [195]:
a=[]
for i in range(len(modifiers)):
    h=modifiers[i]
    a.append(h)
M=1
for i in range(5):
    M=M*a[i]
print(M)


0.9


## Final Score — Choquet Integral Aggregation

The final score combines the ten parameter scores using a **Choquet integral** over a non-additive (fuzzy) measure, which captures interaction effects between parameters that a simple weighted sum cannot express.

### Why a Choquet integral?

A weighted sum assumes each parameter contributes independently. In reality, parameters interact:

- **Synergies** — a lead with strong financials (`X1`) and high urgency (`X4`) is worth more than the sum of those scores would suggest: budget plus urgency together signal an imminent close.
- **Redundancies** — high company fit (`X6`) and high lead-source quality (`X7`) partially overlap in what they tell us, so stacking both shouldn't double-count the confidence.

The Choquet integral handles this with a **capacity** (fuzzy measure) `μ` over all subsets of parameters. For tractability, we use a **2-additive capacity**:

- **Singleton capacities** `a_i` — the base importance of each parameter (analogous to weights).
- **Pairwise interaction coefficients** `a_ij` — positive values create synergy (the pair together is worth more), negative values create redundancy (the pair together is worth less).

### Singleton capacities

| Parameter | `a_i` | Rationale |
|---|---|---|
| `X1` — Financial Qualification | 0.155 | Strong single predictor — no budget, no deal. |
| `X2` — Need & Product Fit | 0.125 | Core value proposition alignment. |
| `X3` — Authority & Decision Structure | 0.200 | Highest weight — access to decision-makers is the #1 accelerator. |
| `X4` — Timeline & Urgency | 0.125 | Timing drives pipeline velocity. |
| `X5` — Engagement Behaviour | 0.120 | Behavioural intent signals. |
| `X6` — Company & Market Fit | 0.060 | Important but largely static / pre-qualification. |
| `X7` — Lead Source Quality | 0.030 | Entry quality — diminishes as engagement data accumulates. |
| `X8` — Competitive Landscape | 0.055 | External pressure factor. |
| `X9` — Relationship & Trust | 0.055 | Relationship equity. |
| `X10` — Strategic & Lifetime Value | 0.075 | Long-term account potential. |

### Pairwise interactions

| Pair | Coefficient | Interpretation |
|---|---|---|
| (`X1`, `X4`) Financial + Timeline | +0.025 | Budget + urgency → imminent close (synergy). |
| (`X1`, `X3`) Financial + Authority | +0.020 | Budget holder with decision power → strong (synergy). |
| (`X2`, `X5`) Need + Engagement | +0.015 | Clear need + active engagement → high intent (synergy). |
| (`X3`, `X4`) Authority + Timeline | +0.015 | Decision-maker + deadline → deal momentum (synergy). |
| (`X8`, `X1`) Competitive + Financial | +0.015 | Competitive advantage + budget → defensible deal (synergy). |
| (`X9`, `X10`) Relationship + Strategic | +0.010 | Trust + long-term value → partnership potential (synergy). |
| (`X5`, `X9`) Engagement + Relationship | −0.010 | High engagement from an existing relationship is partly redundant. |
| (`X6`, `X7`) Company Fit + Source | −0.010 | Both measure "right kind of lead" — some overlap. |
| (`X2`, `X6`) Need + Company Fit | −0.005 | Product fit and ICP fit correlate — slight redundancy. |

### Pre-processing before the integral

- Three parameter scores (`X4`, `X6`, `X9` — indices 3, 5, 8) receive an additional sigmoid transform (`x0 = 0.50`, `k = 8.0`) before entering the integral. These tend to cluster in a narrow mid-range, and the sigmoid spreads them for better discrimination.
- All scores are clamped to `[0, 1]` before aggregation.

### Final formula

```
C_μ   = ChoquetIntegral(X_final, a, a_ij)
μ(N)  = mu(full set)          [normalisation constant]
S     = 100 × G × M × (C_μ / μ(N))
```

The result is a score between 0 and 100, where:

- **`G`** (gates) can zero it instantly,
- **`M`** (modifiers) scales it for deal-execution friction, and
- **`C_μ / μ(N)`** is the Choquet-normalised composite of all ten parameters.


In [181]:
def mu(S, a, a_ij):
    value = sum(a[i] for i in S)
    for (i, j), coeff in a_ij.items():
        if i in S and j in S:
            value += coeff
    return value
    
def choquet_integral(scores, a, a_ij):
    n = len(scores)
    sigma = sorted(range(n), key=lambda i: scores[i])
    integral = 0.0
    for rank in range(n):
        coalition = set(sigma[rank:])
        mu_val = mu(coalition, a, a_ij)
        if rank == 0:
            delta = scores[sigma[rank]]
        else:
            delta = scores[sigma[rank]] - scores[sigma[rank - 1]]
        integral += delta * mu_val
    return integral
    
X_all = [X1, X2, X3, X4, X5, X6, X7, X8, X9, X10]

sigmoid_indices = {3, 5, 8}  
X_final = [
    sigmoid_transform(x, x0=0.50, k=8.0) if i in sigmoid_indices else x
    for i, x in enumerate(X_all)
]
X_final = [max(0.0, min(1.0, x)) for x in X_final]

a_vals = {
    0: 0.155, 1: 0.125, 2: 0.200, 3: 0.125, 4: 0.120,  
    5: 0.060, 6: 0.030, 7: 0.055, 8: 0.055, 9: 0.075,   
}

a_ij_vals = {
    (0, 3): +0.025,
    (0, 2): +0.020, 
    (1, 4): +0.015,  
    (2, 3): +0.015,  
    (7, 0): +0.015, 
    (8, 9): +0.010, 
    (4, 8): -0.010,  
    (5, 6): -0.010,   
    (1, 5): -0.005,   
}

C_mu = choquet_integral(X_final, a_vals, a_ij_vals)
mu_N = mu(set(range(10)), a_vals, a_ij_vals)

S = 100.0 * G * M * (C_mu / mu_N)

print(f"\n{'='*100}")
print(f"  FINAL LEAD SCORE: {S:.2f} / 100")
print(f"{'='*100}")



  FINAL LEAD SCORE: 65.91 / 100
